### Installing Libraries

In [ ]:
!pip install PyPDF2
!pip install pdfplumber
!pip install sentence-transformers nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3

In [ ]:
import os
import json
from google.colab import userdata
from google.colab import drive
import pdfplumber
from operator import itemgetter
from pathlib import Path
import PyPDF2
import pandas as pd
import openai
import numpy as np

### Mounting Drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


### Changing Directory

In [ ]:
os.chdir('/content/drive/MyDrive/GenAI_Course_Master/HelpMate_AI')

In [ ]:
print("Current Dir - ", os.getcwd())

Current Dir -  /content/drive/MyDrive/GenAI_Course_Master/HelpMate_AI


In [ ]:
!find /content/drive -name "HelpMate_AI.ipynb"

/content/drive/MyDrive/GenAI_Course_Master/HelpMate_AI/HelpMate_AI.ipynb


In [ ]:
!ls

'Copy of HelpMate_AI'   Principal-Sample-Life-Insurance-Policy.pdf
 HelpMate_AI.ipynb


### Adding OpenAI Key

In [ ]:
api_key = userdata.get('OpenAI')

### Code to Chunk and Embed PDF using SentenceTransformers

#### Extracting Text From PDF

In [ ]:
pdf_path = Path('/content/drive/MyDrive/GenAI_Course_Master/HelpMate_AI/Principal-Sample-Life-Insurance-Policy.pdf')

In [ ]:
pdf_path

PosixPath('/content/drive/MyDrive/GenAI_Course_Master/HelpMate_AI/Principal-Sample-Life-Insurance-Policy.pdf')

In [ ]:
# Function to check whether a word is present in a table or not for segregation of regular text and tables
def check_bboxes(word, table_bbox):
    # Check whether word is inside a table bbox.
    l = word['x0'], word['top'], word['x1'], word['bottom']
    r = table_bbox
    return l[0] > r[0] and l[1] > r[1] and l[2] < r[2] and l[3] < r[3]

In [ ]:
def extract_text_from_pdf(pdf_path):
    extracted_data = []
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for i, page in enumerate(pdf_reader.pages):
            text = page.extract_text()
            extracted_data.append([i + 1, text if text else ""])
    return extracted_data

In [ ]:
# Function to Extract text from PDF
def extract_text_from_pdf2(pdf_path):
    p = 0               # Storing the iteration loop storing page number depending on the text
    full_text = []      # A empty list to "full_text" to store all the text files

    with pdfplumber.open(pdf_path) as pdf:
      for page in pdf.pages:
        page_no = f"Page {p+1}"
        text = page.extract_text()

        tables = page.find_tables()
        table_bboxes = [i.bbox for i in tables]
        tables = [{'table': i.extract(), 'top':i.bbox[1]} for i in tables]
        non_table_words = [word for word in page.extract_words() if not any([check_bboxes(word, table_bbox) for table_bbox in table_bboxes])]
        lines = []

        for cluster in pdfplumber.utils.cluster_objects(non_table_words + tables, itemgetter('top'), tolerance = 5):
          if 'text' in cluster[0]:
            try:
              lines.append(' '.join([word['text'] for word in cluster]))
            except KeyError:
              pass
          elif 'table' in cluster[0]:
            lines.append(json.dumps(cluster[0]['table']))
        cleaned_text = ' '.join(filter(None, lines)).strip()
        full_text.append([page_no, cleaned_text])
        p += 1
    return full_text

In [ ]:
# Process the PDF
print(f"...Processing {pdf_path.name}")

...Processing Principal-Sample-Life-Insurance-Policy.pdf


#### Chunking Text for better embedding

In [ ]:
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

def chunk_text(text_input, chunk_size=500, overlap=50, strategy="sliding"):
    """
    Chunk long text into smaller pieces for embedding.

    Parameters:
        text_input: str or List[str] — The input text or list of texts (e.g. from pages)
        chunk_size: int — Max number of words per chunk
        overlap: int — Number of overlapping words between chunks
        strategy: str — 'sliding' or 'sentence'

    Returns:
        List[str] — Chunked text pieces
    """
    chunks = []

    # Combine if list of strings
    if isinstance(text_input, list):
        text = " ".join(text_input)
    else:
        text = text_input

    if strategy == "sliding":
        words = text.split()
        for i in range(0, len(words), chunk_size - overlap):
            chunk = " ".join(words[i:i + chunk_size])
            chunks.append(chunk)

    elif strategy == "sentence":
        sentences = sent_tokenize(text)
        current_chunk = ""
        for sentence in sentences:
            if len(current_chunk.split()) + len(sentence.split()) <= chunk_size:
                current_chunk += " " + sentence
            else:
                chunks.append(current_chunk.strip())
                current_chunk = sentence
        if current_chunk:
            chunks.append(current_chunk.strip())

    else:
        raise ValueError("Invalid strategy. Choose 'sliding' or 'sentence'.")

    return chunks

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
# 1. Extract text (including tables) from the PDF
extracted_text = extract_text_from_pdf(pdf_path)

# 2. Convert the extracted data into a DataFrame
extracted_text_df = pd.DataFrame(extracted_text, columns=['Page No.', 'Page_Text'])

# 3. Add a column for the document name
extracted_text_df['Document Name'] = Path(pdf_path).name

# 4. Append to a list (if you plan to process multiple PDFs)
data = []
data.append(extracted_text_df)

# 5. Display confirmation and preview
print(f"✅ Finished processing {pdf_path}")
print(extracted_text_df.head())  # Show first 5 rows

✅ Finished processing /content/drive/MyDrive/GenAI_Course_Master/HelpMate_AI/Principal-Sample-Life-Insurance-Policy.pdf
   Page No.                                          Page_Text  \
0         1   \n  \n \n \nGROUP POLICY FOR:  \nRHODE ISLAND...   
1         2   \n \n \n  \n \n \n \n  \n \n \n \n   \n \n \n...   
2         3   \n \n \n  \n  \nGC 806 VAL   \n POLICY RIDER ...   
3         4   \n \n \n  \n \n                     \nThis pa...   
4         5   \n \n \nThis policy has been updated effectiv...   

                                Document Name  
0  Principal-Sample-Life-Insurance-Policy.pdf  
1  Principal-Sample-Life-Insurance-Policy.pdf  
2  Principal-Sample-Life-Insurance-Policy.pdf  
3  Principal-Sample-Life-Insurance-Policy.pdf  
4  Principal-Sample-Life-Insurance-Policy.pdf  


In [ ]:
# 1. Extract text (including tables) from the PDF
extracted_text = extract_text_from_pdf2(pdf_path)

# 2. Convert the extracted data into a DataFrame
extracted_text_df = pd.DataFrame(extracted_text, columns=['Page No.', 'Page_Text'])

# 3. Add a column for the document name
extracted_text_df['Document Name'] = Path(pdf_path).name

# 4. Append to a list (if you plan to process multiple PDFs)
data = []
data.append(extracted_text_df)

# 5. Display confirmation and preview
print(f"✅ Finished processing {pdf_path}")
print(extracted_text_df.head())  # Show first 5 rows


✅ Finished processing /content/drive/MyDrive/GenAI_Course_Master/HelpMate_AI/Principal-Sample-Life-Insurance-Policy.pdf
  Page No.                                          Page_Text  \
0   Page 1  DOROTHEA GLAUSE S655 RHODE ISLAND JOHN DOE 01/...   
1   Page 2                 This page left blank intentionally   
2   Page 3  POLICY RIDER GROUP INSURANCE POLICY NO: S655 C...   
3   Page 4                 This page left blank intentionally   
4   Page 5  PRINCIPAL LIFE INSURANCE COMPANY (called The P...   

                                Document Name  
0  Principal-Sample-Life-Insurance-Policy.pdf  
1  Principal-Sample-Life-Insurance-Policy.pdf  
2  Principal-Sample-Life-Insurance-Policy.pdf  
3  Principal-Sample-Life-Insurance-Policy.pdf  
4  Principal-Sample-Life-Insurance-Policy.pdf  


In [ ]:
insurance_pdf_data = pd.concat(data, ignore_index=True)

In [ ]:
insurance_pdf_data.head()

,Page No.,Page_Text,Document Name
0,Page 1,DOROTHEA GLAUSE S655 RHODE ISLAND JOHN DOE 01/...,Principal-Sample-Life-Insurance-Policy.pdf
1,Page 2,This page left blank intentionally,Principal-Sample-Life-Insurance-Policy.pdf
2,Page 3,POLICY RIDER GROUP INSURANCE POLICY NO: S655 C...,Principal-Sample-Life-Insurance-Policy.pdf
3,Page 4,This page left blank intentionally,Principal-Sample-Life-Insurance-Policy.pdf
4,Page 5,PRINCIPAL LIFE INSURANCE COMPANY (called The P...,Principal-Sample-Life-Insurance-Policy.pdf


In [ ]:
insurance_pdf_data['Metadata'] = insurance_pdf_data.apply(
    lambda x: {
        'filing_name': x['Document Name'][:-4],  # Removes ".pdf" from filename
        'Page_No.': x['Page No.']
    },
    axis=1
)

In [ ]:
insurance_pdf_data.head()

,Page No.,Page_Text,Document Name,Metadata
0,Page 1,DOROTHEA GLAUSE S655 RHODE ISLAND JOHN DOE 01/...,Principal-Sample-Life-Insurance-Policy.pdf,{'filing_name': 'Principal-Sample-Life-Insuran...
1,Page 2,This page left blank intentionally,Principal-Sample-Life-Insurance-Policy.pdf,{'filing_name': 'Principal-Sample-Life-Insuran...
2,Page 3,POLICY RIDER GROUP INSURANCE POLICY NO: S655 C...,Principal-Sample-Life-Insurance-Policy.pdf,{'filing_name': 'Principal-Sample-Life-Insuran...
3,Page 4,This page left blank intentionally,Principal-Sample-Life-Insurance-Policy.pdf,{'filing_name': 'Principal-Sample-Life-Insuran...
4,Page 5,PRINCIPAL LIFE INSURANCE COMPANY (called The P...,Principal-Sample-Life-Insurance-Policy.pdf,{'filing_name': 'Principal-Sample-Life-Insuran...


In [ ]:
all_chunks = []

for index, row in extracted_text_df.iterrows():
    page_chunks = chunk_text(row["Page_Text"], chunk_size=500, overlap=50, strategy="sliding")
    for i, chunk in enumerate(page_chunks):
        all_chunks.append({
            "Page_No.": row["Page No."],
            "Chunk_No.": f"{row['Page No.']}_Chunk_{i+1}",
            "Chunk_text": chunk,
            "Document_Name": row["Document Name"]
        })

chunked_df = pd.DataFrame(all_chunks)
print(chunked_df.head())

  Page_No.       Chunk_No.                                         Chunk_text  \
0   Page 1  Page 1_Chunk_1  DOROTHEA GLAUSE S655 RHODE ISLAND JOHN DOE 01/...   
1   Page 2  Page 2_Chunk_1                 This page left blank intentionally   
2   Page 3  Page 3_Chunk_1  POLICY RIDER GROUP INSURANCE POLICY NO: S655 C...   
3   Page 4  Page 4_Chunk_1                 This page left blank intentionally   
4   Page 5  Page 5_Chunk_1  PRINCIPAL LIFE INSURANCE COMPANY (called The P...   

                                Document_Name  
0  Principal-Sample-Life-Insurance-Policy.pdf  
1  Principal-Sample-Life-Insurance-Policy.pdf  
2  Principal-Sample-Life-Insurance-Policy.pdf  
3  Principal-Sample-Life-Insurance-Policy.pdf  
4  Principal-Sample-Life-Insurance-Policy.pdf  


In [ ]:
chunked_df.head()

,Page_No.,Chunk_No.,Chunk_text,Document_Name
0,Page 1,Page 1_Chunk_1,DOROTHEA GLAUSE S655 RHODE ISLAND JOHN DOE 01/...,Principal-Sample-Life-Insurance-Policy.pdf
1,Page 2,Page 2_Chunk_1,This page left blank intentionally,Principal-Sample-Life-Insurance-Policy.pdf
2,Page 3,Page 3_Chunk_1,POLICY RIDER GROUP INSURANCE POLICY NO: S655 C...,Principal-Sample-Life-Insurance-Policy.pdf
3,Page 4,Page 4_Chunk_1,This page left blank intentionally,Principal-Sample-Life-Insurance-Policy.pdf
4,Page 5,Page 5_Chunk_1,PRINCIPAL LIFE INSURANCE COMPANY (called The P...,Principal-Sample-Life-Insurance-Policy.pdf


In [ ]:
chunked_df['Metadata'] = chunked_df.apply(
    lambda x: {
        'filing_name': x['Document_Name'][:-4],  # Removes ".pdf" from filename
        'Page_No.': x['Page_No.']
    },
    axis=1
)

In [ ]:
chunked_df.head()

,Page_No.,Chunk_No.,Chunk_text,Document_Name,Metadata
0,Page 1,Page 1_Chunk_1,DOROTHEA GLAUSE S655 RHODE ISLAND JOHN DOE 01/...,Principal-Sample-Life-Insurance-Policy.pdf,{'filing_name': 'Principal-Sample-Life-Insuran...
1,Page 2,Page 2_Chunk_1,This page left blank intentionally,Principal-Sample-Life-Insurance-Policy.pdf,{'filing_name': 'Principal-Sample-Life-Insuran...
2,Page 3,Page 3_Chunk_1,POLICY RIDER GROUP INSURANCE POLICY NO: S655 C...,Principal-Sample-Life-Insurance-Policy.pdf,{'filing_name': 'Principal-Sample-Life-Insuran...
3,Page 4,Page 4_Chunk_1,This page left blank intentionally,Principal-Sample-Life-Insurance-Policy.pdf,{'filing_name': 'Principal-Sample-Life-Insuran...
4,Page 5,Page 5_Chunk_1,PRINCIPAL LIFE INSURANCE COMPANY (called The P...,Principal-Sample-Life-Insurance-Policy.pdf,{'filing_name': 'Principal-Sample-Life-Insuran...


In [ ]:
print(len(insurance_pdf_data))
print(len(chunked_df))

64
66


### Chroma Database

In [ ]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 1.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 97.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 569.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.2/196.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 8.3 MB/s et

In [ ]:
# Importing the OpenAI Embedding Function into Chroma
import chromadb
from chromadb.config import Settings
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

In [ ]:
chromadb_data_path = '/content/drive/MyDrive/GenAI_Course_Master/HelpMate_AI/ChromaDB_presistpath_Chuncking'

In [ ]:
import logging
logging.getLogger("chromadb.telemetry").setLevel(logging.CRITICAL)

In [ ]:
chroma_client = chromadb.PersistentClient(path=chromadb_data_path,settings=Settings(anonymized_telemetry=False))

In [ ]:
model = 'text-embedding-ada-002'
embedding_functions = OpenAIEmbeddingFunction(api_key=api_key, model_name=model)

In [ ]:
insurance_collection = chroma_client.get_or_create_collection(name='Insurance_RAG', embedding_function=embedding_functions)

In [ ]:
insurance_collection

Collection(name=Insurance_RAG)

In [ ]:
documents_list = chunked_df['Chunk_text'].tolist()
metadata_list = chunked_df['Metadata'].tolist()

In [ ]:
insurance_collection.add(
    documents = documents_list,
    ids = [str(i) for i in range(0, len(documents_list))],
    metadatas = metadata_list
)

In [ ]:
insurance_collection

Collection(name=Insurance_RAG)

In [ ]:
insurance_collection.peek(2)

{'ids': ['0', '1'],
 'embeddings': array([[-0.02251517,  0.01869307, -0.02726344, ..., -0.03691652,
          0.00289266, -0.00121071],
        [-0.01595563,  0.00016479,  0.0027096 , ..., -0.01704412,
          0.0027544 , -0.02559273]]),
 'documents': ['DOROTHEA GLAUSE S655 RHODE ISLAND JOHN DOE 01/01/2014 711 HIGH STREET GEORGE RI 02903 GROUP POLICY FOR: RHODE ISLAND JOHN DOE ALL MEMBERS Group Member Life Insurance Print Date: 07/16/2014',
  'This page left blank intentionally'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'Page_No.': 'Page 1',
   'filing_name': 'Principal-Sample-Life-Insurance-Policy'},
  {'filing_name': 'Principal-Sample-Life-Insurance-Policy',
   'Page_No.': 'Page 2'}]}

In [ ]:
cache_collection = chroma_client.get_or_create_collection(name='cache_Insurance_RAG', embedding_function=embedding_functions)

In [ ]:
cache_collection.peek(2)

{'ids': [],
 'embeddings': array([], dtype=float64),
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': []}

### Reading User Query

In [ ]:
query = input()

Death benifits


In [ ]:
query

'Death benifits'

In [ ]:
results = insurance_collection.query(
    query_texts=query,
    n_results=2
)

In [ ]:
results

{'ids': [['48', '7']],
 'embeddings': None,
 'documents': [["M ember's death, the Death Benefits Payable may be withheld until additional information has been received or the trial has been held. If a Member who was insured dies within the 31-day individual purchase period described in PART III, Section F, The Principal will pay his or her beneficiary the individual policy amount, if any, the Member had the right to purchase. No payment will be made before The Principal receives Written proof of the Member's death. Article 3 - Beneficiary A beneficiary should be named at the time a Member applies or enrolls under this Group Policy. A Member may name or later change a named beneficiary by sending a Written request to The Principal. A change will not be effective until recorded by The Principal. Once recorded, the change will apply as of the date the request was Signed. If The Principal properly pays any benefit before a change request is received, that payment may not be contested. Furt

In [ ]:
for key, value in results.items():
  print(key)

ids
embeddings
documents
uris
included
data
metadatas
distances


In [ ]:
for key, value in results.items():
  print(key)
  print(value)

ids
[['48', '7']]
embeddings
None
documents
[["M ember's death, the Death Benefits Payable may be withheld until additional information has been received or the trial has been held. If a Member who was insured dies within the 31-day individual purchase period described in PART III, Section F, The Principal will pay his or her beneficiary the individual policy amount, if any, the Member had the right to purchase. No payment will be made before The Principal receives Written proof of the Member's death. Article 3 - Beneficiary A beneficiary should be named at the time a Member applies or enrolls under this Group Policy. A Member may name or later change a named beneficiary by sending a Written request to The Principal. A change will not be effective until recorded by The Principal. Once recorded, the change will apply as of the date the request was Signed. If The Principal properly pays any benefit before a change request is received, that payment may not be contested. Further: a. The na

In [ ]:
for key, val in results.items():
  if key not in ['embedding', 'uris', 'data']:
    for i in range (10):
      print(str(key)+str(i))

ids0
ids1
ids2
ids3
ids4
ids5
ids6
ids7
ids8
ids9
embeddings0
embeddings1
embeddings2
embeddings3
embeddings4
embeddings5
embeddings6
embeddings7
embeddings8
embeddings9
documents0
documents1
documents2
documents3
documents4
documents5
documents6
documents7
documents8
documents9
included0
included1
included2
included3
included4
included5
included6
included7
included8
included9
metadatas0
metadatas1
metadatas2
metadatas3
metadatas4
metadatas5
metadatas6
metadatas7
metadatas8
metadatas9
distances0
distances1
distances2
distances3
distances4
distances5
distances6
distances7
distances8
distances9


In [ ]:
# Query the cache collection againts the user query and return the top results
cache_results = cache_collection.query(
    query_texts=query,
    n_results=1
)

In [ ]:
cache_results

{'ids': [[]],
 'embeddings': None,
 'documents': [[]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[]],
 'distances': [[]]}

In [ ]:
cache_results['distances'][0]

[]

### Implementing Cache in Semantic Search

In [ ]:
from functools import cache
threshold = 0.2 # Setting a threshold for cache search

ids = []
documents = []
distances = []
metadatas = []
results_df = pd.DataFrame()

# If the distance is greater than the threshold, then return the results from the main collection.

if cache_results['distances'][0] == [] or cache_results['distances'][0][0] > threshold:
  # Query the collection against the user query and return the top 10 results
  results = insurance_collection.query(
      query_texts=query,
      n_results=10
  )

  # Storing the query in cache_collection as document w.r.t to ChromaDB so that it can be embedded and searched against later
  # Storing retrieved text, ids, distances and metadatas in cache_collection as metadatas, so that they can be fetched easily if a query indeed matches to a query in cache

  keys = []
  values = []

  for key, value in results.items():
    if key not in ['embedding', 'uris', 'data'] and value is not None:
        for i, item in enumerate(value[0]):
            keys.append(f"{key}{i}")
            values.append(str(item))

  cache_collection.add(
      documents = [query],
      ids=[query],
      metadatas=dict(zip(keys, values))
  )

  print("Result not found in cache. Returning results from main collection.")

  result_dict = {'ids': results['ids'][0], 'documents': results['documents'][0], 'distances': results['distances'][0], 'metadatas': results['metadatas'][0]}
  results_df = pd.DataFrame(result_dict)
  results_df

# If the distance is, however, less than the threshold, you can return the results from cache

elif cache_results['distances'][0][0] < threshold:
  cache_results_dict = cache_results['metadatas'][0][0]

  # Loop through each inner list and then through the dictionary
  for key, value in cache_results_dict.items():
    if 'documents' in key:
      documents.append(value)
    elif 'ids' in key:
      ids.append(value)
    elif 'distances' in key:
      distances.append(value)
    elif 'metadatas' in key:
      metadatas.append(value)
  print("Result found in cache. Returning results from cache.")

  # Creating a Dataframe
  results_df = pd.DataFrame({
        'IDs': ids,
        'Documents': documents,
        'Distances': distances,
        'Metadatas': metadatas
      })


Result not found in cache. Returning results from main collection.


In [ ]:
results_df

,ids,documents,distances,metadatas
0,48,"M ember's death, the Death Benefits Payable ma...",0.289319,"{'Page_No.': 'Page 47', 'filing_name': 'Princi..."
1,7,Section A - Member Life Insurance Schedule of ...,0.314246,"{'Page_No.': 'Page 8', 'filing_name': 'Princip..."
2,61,I f a Dependent who was insured dies during th...,0.320214,"{'Page_No.': 'Page 60', 'filing_name': 'Princi..."
3,55,"f . claim requirements listed in PART IV, Sect...",0.320991,"{'Page_No.': 'Page 54', 'filing_name': 'Princi..."
4,58,% of Scheduled Covered Loss Benefit Loss of Sp...,0.322053,"{'Page_No.': 'Page 57', 'filing_name': 'Princi..."
5,32,(1) marriage or establishment of a Civil Union...,0.326675,"{'Page_No.': 'Page 32', 'filing_name': 'Princi..."
6,56,Exposure Exposure to the elements will be pres...,0.343172,{'filing_name': 'Principal-Sample-Life-Insuran...
7,52,Coverage During Disability will cease on the e...,0.345069,{'filing_name': 'Principal-Sample-Life-Insuran...
8,46,(1) If termination is as described in b. (1) a...,0.350154,{'filing_name': 'Principal-Sample-Life-Insuran...
9,60,Section C - Dependent Life Insurance Article 1...,0.351510,"{'Page_No.': 'Page 59', 'filing_name': 'Princi..."


In [ ]:
cache_results = cache_collection.query(
    query_texts=query,
    n_results=1
)
cache_results

{'ids': [['Death benifits']],
 'embeddings': None,
 'documents': [['Death benifits']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'distances5': '0.3266754150390625',
    'included1': 'e',
    'documents4': '% of Scheduled Covered Loss Benefit Loss of Speech and/or Hearing Speech and Hearing 100% Speech or Hearing 50% Hearing in One Ear 25% Loss must be determined by a Physician to be permanent, complete, and irreversible. Total payment for all losses that result from the same accident will not exceed the Scheduled Benefit (or approved amount, if applicable). Payment for Loss will be to the Member. For this benefit, the term "Loss" means a total and irrevocable Loss of speech or hearing, which has continued for 12 consecutive months. Article 7 - Repatriation Benefit If a benefit is paid under this Section B for loss of the Member\'s life and death occurred at least 100 miles away from the Member\'s permanent place of residence, a

### Working with Cross-Encoder Tranformers

In [ ]:
!pip install sentence_transformers

In [ ]:
from sentence_transformers import CrossEncoder, util

In [ ]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
results_df.head()

,ids,documents,distances,metadatas
0,48,"M ember's death, the Death Benefits Payable ma...",0.289319,"{'Page_No.': 'Page 47', 'filing_name': 'Princi..."
1,7,Section A - Member Life Insurance Schedule of ...,0.314246,"{'Page_No.': 'Page 8', 'filing_name': 'Princip..."
2,61,I f a Dependent who was insured dies during th...,0.320214,"{'Page_No.': 'Page 60', 'filing_name': 'Princi..."
3,55,"f . claim requirements listed in PART IV, Sect...",0.320991,"{'Page_No.': 'Page 54', 'filing_name': 'Princi..."
4,58,% of Scheduled Covered Loss Benefit Loss of Sp...,0.322053,"{'Page_No.': 'Page 57', 'filing_name': 'Princi..."


In [ ]:
# Testing cross-endocder
# Input (query, response) pairs for each of the top 20 responses received from the semantic search to the cross encoder
# Generate the cross_encoder scores for these pairs

cross_inputs = [[query, response] for response in results_df['documents']]
cross_rerank_scores = cross_encoder.predict(cross_inputs)

In [ ]:
 #Store the rerank_scores in results_df
results_df['Reranked_scores'] = cross_rerank_scores

In [ ]:
# Return the top 3 results from semantic search
top_3_semantic = results_df.sort_values(by='distances')
top_3_semantic[:3]

,ids,documents,distances,metadatas,Reranked_scores
0,48,"M ember's death, the Death Benefits Payable ma...",0.289319,"{'Page_No.': 'Page 47', 'filing_name': 'Princi...",-1.901206
1,7,Section A - Member Life Insurance Schedule of ...,0.314246,"{'Page_No.': 'Page 8', 'filing_name': 'Princip...",-5.409644
2,61,I f a Dependent who was insured dies during th...,0.320214,"{'Page_No.': 'Page 60', 'filing_name': 'Princi...",-3.535415


In [ ]:
# Return the top 3 results after reranking

top_3_rerank = results_df.sort_values(by='Reranked_scores', ascending=False)
top_3_rerank[:3]
top_3_RAG = top_3_rerank[["documents", "metadatas"]][:3]

In [ ]:
top_3_RAG

,documents,metadatas
0,"M ember's death, the Death Benefits Payable ma...","{'Page_No.': 'Page 47', 'filing_name': 'Princi..."
2,I f a Dependent who was insured dies during th...,"{'Page_No.': 'Page 60', 'filing_name': 'Princi..."
1,Section A - Member Life Insurance Schedule of ...,"{'Page_No.': 'Page 8', 'filing_name': 'Princip..."


In [ ]:
top_3_RAG['documents'].iloc[0]

"M ember's death, the Death Benefits Payable may be withheld until additional information has been received or the trial has been held. If a Member who was insured dies within the 31-day individual purchase period described in PART III, Section F, The Principal will pay his or her beneficiary the individual policy amount, if any, the Member had the right to purchase. No payment will be made before The Principal receives Written proof of the Member's death. Article 3 - Beneficiary A beneficiary should be named at the time a Member applies or enrolls under this Group Policy. A Member may name or later change a named beneficiary by sending a Written request to The Principal. A change will not be effective until recorded by The Principal. Once recorded, the change will apply as of the date the request was Signed. If The Principal properly pays any benefit before a change request is received, that payment may not be contested. Further: a. The naming of a new beneficiary in an application fo

In [ ]:
retrieved = top_3_RAG['documents'].iloc[0]

In [ ]:
query

'Death benifits'

### Generative Search Using OpenAI

In [ ]:
messages = [
  {
    "role": "system",
    "content": (
      "You are a highly reliable and detail-oriented AI assistant trained to help users understand insurance-related information. "
      "Your primary role is to answer queries by strictly using the facts retrieved from official insurance documents, such as filing reports, policy summaries, or regulatory filings. "
      "Do not generate information beyond what is retrieved. If the information is not in the provided documents, clearly state that you cannot answer. "
      "Maintain a professional, clear, and informative tone at all times."
    )
  },
  {
    "role": "user",
    "content": (
      f"{query}\n\n"
      f"Use only the retrieved content from the Principal-Sample-Life-Insurance-Policy report:\n\n{retrieved}"
    )
  }
]


In [ ]:
client_ai = openai.OpenAI(api_key=api_key)
response = client_ai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages
      )
response.choices[0].message.content

"In the provided excerpt from the Principal-Sample-Life-Insurance-Policy report, the information related to Death Benefits is as follows:\n\n1. Death Benefits Payable:\n- In the event of a Member's death, the Death Benefits Payable may be withheld until additional information is received or the trial has been held.\n- If a Member who was insured dies within the 31-day individual purchase period, The Principal will pay the beneficiary the individual policy amount that the Member had the right to purchase.\n- No payment will be made until The Principal receives written proof of the Member's death.\n\n2. Beneficiary:\n- A beneficiary should be named at the time a Member applies or enrolls under the Group Policy.\n- A Member may name or change a beneficiary by sending a written request to The Principal.\n- The change will only be effective once recorded by The Principal, and it will apply as of the date the request was signed.\n- If a benefit is properly paid before a change request is rec

In [ ]:
print(response.choices[0].message.content)

In the provided excerpt from the Principal-Sample-Life-Insurance-Policy report, the information related to Death Benefits is as follows:

1. Death Benefits Payable:
- In the event of a Member's death, the Death Benefits Payable may be withheld until additional information is received or the trial has been held.
- If a Member who was insured dies within the 31-day individual purchase period, The Principal will pay the beneficiary the individual policy amount that the Member had the right to purchase.
- No payment will be made until The Principal receives written proof of the Member's death.

2. Beneficiary:
- A beneficiary should be named at the time a Member applies or enrolls under the Group Policy.
- A Member may name or change a beneficiary by sending a written request to The Principal.
- The change will only be effective once recorded by The Principal, and it will apply as of the date the request was signed.
- If a benefit is properly paid before a change request is received, that 

### Final Fuction to Stich everything together

In [ ]:
def retrieve_and_generate(query,
                          main_collection=insurance_collection,
                          cache_collection=cache_collection,
                          embed_model=embedding_functions,
                          reranker_model=cross_encoder,
                          llm_client=client_ai,
                          k_main=10,
                          k_final=3,
                          threshold=0.2):
    """
    End-to-end RAG pipeline for insurance chatbot:
    1. Check cache.
    2. If not found or far, search main collection.
    3. Store result in cache if needed.
    4. Rerank results using cross-encoder.
    5. Generate answer using OpenAI GPT.
    """

    print(f"\n Query received: {query}")

    query_embedding = client_ai.embeddings.create(
    model="text-embedding-ada-002",
    input=[query]
    ).data[0].embedding

    # Step 2: Check cache
    cache_results = cache_collection.query(query_embeddings=[query_embedding], n_results=1)
    retrieved_from_cache = False

    if cache_results['distances'][0] and cache_results['distances'][0][0] < threshold:
        print("\n User Query Found in cache.")
        cache_results_dict = cache_results['metadatas'][0][0]
        documents = [v for k, v in cache_results_dict.items() if k.startswith('documents')]
        ids = [v for k, v in cache_results_dict.items() if k.startswith('ids')]
        distances = [v for k, v in cache_results_dict.items() if k.startswith('distances')]
        metadatas = [v for k, v in cache_results_dict.items() if k.startswith('metadatas')]
        results_df = pd.DataFrame({'IDs': ids, 'documents': documents, 'Distances': distances, 'Metadatas': metadatas})
        retrieved_from_cache = True
    else:
        print("\n User Query Not found in cache. Querying main collection...")
        results = main_collection.query(query_embeddings=[query_embedding], n_results=k_main)
        results_df = pd.DataFrame({
            'IDs': results['ids'][0],
            'documents': results['documents'][0],
            'Distances': results['distances'][0],
            'Metadatas': results['metadatas'][0]
        })

        # Add to cache
        keys, values = [], []
        for key, value in results.items():
            if key not in ['embedding', 'uris', 'data'] and value is not None:
                for i, item in enumerate(value[0]):
                    keys.append(f"{key}{i}")
                    values.append(str(item))
        cache_collection.add(
            documents=[query],
            ids=[query],
            metadatas=dict(zip(keys, values))
        )

    # Step 3: Rerank using cross-encoder
    print("\n Reranking top results...")
    cross_inputs = [[query, doc] for doc in results_df['documents']]
    cross_scores = reranker_model.predict(cross_inputs)
    reranked = sorted(zip(results_df['documents'], cross_scores), key=lambda x: x[1], reverse=True)
    top_docs = [doc for doc, _ in reranked[:k_final]]

    # Step 4: LLM prompt generation
    retrieved_text = "\n\n".join(top_docs)
    messages = [
        {
            "role": "system",
            "content": (
                "You are a highly reliable and detail-oriented AI assistant trained to help users understand insurance-related information. "
                "Your primary role is to answer queries by strictly using the facts retrieved from official insurance documents, such as filing reports, policy summaries, or regulatory filings. "
                "Do not generate information beyond what is retrieved. If the information is not in the provided documents, clearly state that you cannot answer. "
                "Maintain a professional, clear, and informative tone at all times."
            )
        },
        {
            "role": "user",
            "content": (
                f"{query}\n\n"
                f"Use only the retrieved content from the insurance policy report:\n\n{retrieved_text}"
            )
        }
    ]

    # Step 5: Generate answer
    try:
        response = llm_client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=messages
        )
        answer = response.choices[0].message.content
        print("\n Answer:\n", answer)
        return answer
    except Exception as e:
        print(f"\n Error generating response: {e}")
        return "\n Sorry, something went wrong while generating the response."


## Query

🔹 Query 1:
"What is the claim process for nominees of the policy?"

✅ Tests your system's ability to retrieve procedural content.

🔹 Query 2:
"Are there any exclusions or conditions under which the insurance benefits are not paid?"

✅ Useful to validate retrieval of exception clauses and fine print.

🔹 Query 3:
"How is the policy surrender value calculated, and when is it applicable?"

✅ Checks the pipeline's effectiveness in explaining financial terms from policy documents.



In [ ]:
query = input("💬 Enter your insurance-related query: ").strip()
print(f"\n🔍 Query received: {query}")
results_df.head(3)  # after retrieval, before reranking

💬 Enter your insurance-related query: What is the claim process for nominees of the policy?

🔍 Query received: What is the claim process for nominees of the policy?


,ids,documents,distances,metadatas,Reranked_scores
0,48,"M ember's death, the Death Benefits Payable ma...",0.289319,"{'Page_No.': 'Page 47', 'filing_name': 'Princi...",-1.901206
1,7,Section A - Member Life Insurance Schedule of ...,0.314246,"{'Page_No.': 'Page 8', 'filing_name': 'Princip...",-5.409644
2,61,I f a Dependent who was insured dies during th...,0.320214,"{'Page_No.': 'Page 60', 'filing_name': 'Princi...",-3.535415


In [ ]:
query = input("Enter your insurance-related query: ")
response = retrieve_and_generate(query)

Enter your insurance-related query: What is the claim process for nominees of the policy?

 Query received: What is the claim process for nominees of the policy?

 User Query Not found in cache. Querying main collection...

 Reranking top results...

 Answer:
 The claim process for nominees of the policy is as follows:

1. **Notice of Claim**: The nominee needs to provide written notice to The Principal within 20 days after the date of the loss for which the claim is being made. However, if notice is given as soon as reasonably possible, it will not invalidate or reduce any claim.

2. **Claim Forms**: The Principal will provide appropriate claim forms upon receiving notice of the claim. If the forms are not provided within 15 days, the person can submit written proof covering the occurrence, character, and extent of the loss within the specified time for filing proof of loss.

3. **Proof of Loss**: Written proof of loss must be sent to The Principal within 90 days after the date of the

In [ ]:
query = input("💬 Enter your insurance-related query: ").strip()
print(f"\n🔍 Query received: {query}")
results_df.head(3)  # after retrieval, before reranking

💬 Enter your insurance-related query: Are there any exclusions or conditions under which the insurance benefits are not paid?

🔍 Query received: Are there any exclusions or conditions under which the insurance benefits are not paid?


,ids,documents,distances,metadatas,Reranked_scores
0,48,"M ember's death, the Death Benefits Payable ma...",0.289319,"{'Page_No.': 'Page 47', 'filing_name': 'Princi...",-1.901206
1,7,Section A - Member Life Insurance Schedule of ...,0.314246,"{'Page_No.': 'Page 8', 'filing_name': 'Princip...",-5.409644
2,61,I f a Dependent who was insured dies during th...,0.320214,"{'Page_No.': 'Page 60', 'filing_name': 'Princi...",-3.535415


In [ ]:
query = input("Enter your insurance-related query: ")
response = retrieve_and_generate(query)

Enter your insurance-related query: Are there any exclusions or conditions under which the insurance benefits are not paid?

 Query received: Are there any exclusions or conditions under which the insurance benefits are not paid?

 User Query Not found in cache. Querying main collection...

 Reranking top results...

 Answer:
 Based on the information provided in the insurance policy report, there are specific exclusions and conditions under which insurance benefits may not be paid. Here are some of the key points:

1. If a beneficiary is found guilty of the Member's death, they may be disqualified from receiving any benefit due. In such a case, payment may be made to a contingent beneficiary or to the executor or administrator of the Member's estate.

2. No benefits will be paid for any disability that results from willful self-injury or self-destruction, war or act of war, or voluntary participation in activities such as assault, felony, criminal activity, insurrection, or riot.

3. 

In [ ]:
query = input("💬 Enter your insurance-related query: ").strip()
print(f"\n🔍 Query received: {query}")
results_df.head(3)  # after retrieval, before reranking

💬 Enter your insurance-related query: How is the policy surrender value calculated, and when is it applicable?

🔍 Query received: How is the policy surrender value calculated, and when is it applicable?


,ids,documents,distances,metadatas,Reranked_scores
0,48,"M ember's death, the Death Benefits Payable ma...",0.289319,"{'Page_No.': 'Page 47', 'filing_name': 'Princi...",-1.901206
1,7,Section A - Member Life Insurance Schedule of ...,0.314246,"{'Page_No.': 'Page 8', 'filing_name': 'Princip...",-5.409644
2,61,I f a Dependent who was insured dies during th...,0.320214,"{'Page_No.': 'Page 60', 'filing_name': 'Princi...",-3.535415


In [ ]:
query = input("Enter your insurance-related query: ")
response = retrieve_and_generate(query)

Enter your insurance-related query: How is the policy surrender value calculated, and when is it applicable?

 Query received: How is the policy surrender value calculated, and when is it applicable?

 User Query Not found in cache. Querying main collection...

 Reranking top results...

 Answer:
 The information provided in the insurance policy report does not specifically address how the policy surrender value is calculated or when it is applicable. As a result, I cannot provide details on the calculation or applicability of the policy surrender value based on the retrieved content. If you require specific details on this matter, I recommend contacting the insurance provider directly for accurate information regarding the policy surrender value.


In [ ]:
query = input("💬 Enter your insurance-related query: ").strip()
print(f"\n🔍 Query received: {query}")
results_df.head(3)  # after retrieval, before reranking

💬 Enter your insurance-related query: What are the eligibility criteria for dependent life insurance under this policy?

🔍 Query received: What are the eligibility criteria for dependent life insurance under this policy?


,ids,documents,distances,metadatas,Reranked_scores
0,48,"M ember's death, the Death Benefits Payable ma...",0.289319,"{'Page_No.': 'Page 47', 'filing_name': 'Princi...",-1.901206
1,7,Section A - Member Life Insurance Schedule of ...,0.314246,"{'Page_No.': 'Page 8', 'filing_name': 'Princip...",-5.409644
2,61,I f a Dependent who was insured dies during th...,0.320214,"{'Page_No.': 'Page 60', 'filing_name': 'Princi...",-3.535415


In [ ]:
query = input("Enter your insurance-related query: ")
response = retrieve_and_generate(query)

Enter your insurance-related query: What are the eligibility criteria for dependent life insurance under this policy?

 Query received: What are the eligibility criteria for dependent life insurance under this policy?

 User Query Not found in cache. Querying main collection...

 Reranking top results...

 Answer:
 The eligibility criteria for Dependent Life Insurance under this policy include the following:

1. A person will be eligible for Dependent Life Insurance on the latest of:
   a. The date the person is eligible for Member Life Insurance;
   b. The date the person first acquires a Dependent;
   c. The date the person enters a class for which Dependent Life Insurance is provided under this Group Policy;
   d. The date Dependent Life Insurance is added to this Group Policy.

2. If a Member's Dependent is employed and covered under group term life coverage by the Dependent's employer, the date such coverage terminates due to the Dependent no longer being eligible will be consider

In [ ]:
query = input("💬 Enter your insurance-related query: ").strip()
print(f"\n🔍 Query received: {query}")
results_df.head(3)  # after retrieval, before reranking

💬 Enter your insurance-related query: Can a member convert their group life insurance to an individual policy after termination? What is the process?

🔍 Query received: Can a member convert their group life insurance to an individual policy after termination? What is the process?


,ids,documents,distances,metadatas,Reranked_scores
0,48,"M ember's death, the Death Benefits Payable ma...",0.289319,"{'Page_No.': 'Page 47', 'filing_name': 'Princi...",-1.901206
1,7,Section A - Member Life Insurance Schedule of ...,0.314246,"{'Page_No.': 'Page 8', 'filing_name': 'Princip...",-5.409644
2,61,I f a Dependent who was insured dies during th...,0.320214,"{'Page_No.': 'Page 60', 'filing_name': 'Princi...",-3.535415


In [ ]:
query = input("Enter your insurance-related query: ")
response = retrieve_and_generate(query)

Enter your insurance-related query: Can a member convert their group life insurance to an individual policy after termination? What is the process?

 Query received: Can a member convert their group life insurance to an individual policy after termination? What is the process?

 User Query Not found in cache. Querying main collection...

 Reranking top results...

 Answer:
 Yes, a member can convert their group life insurance to an individual policy under certain conditions as outlined in the insurance policy. Here is the process:

1. **Purchase Qualification**: The member will qualify for individual purchase if their insurance under the Group Policy terminates due to specific reasons listed below:
    - The member's total Life Insurance terminates because they end Active Work or are no longer in a class eligible for insurance.
    - After being continuously insured under the Group Policy for at least five years, the member's total Member Life Insurance terminates due to the Group Poli